# BTP Part 2: 3D Face Geometry-Based Face Presentation Attack Detection (PAD)
## Research Execution & Evaluation Notebook (OULU Pilot-100)

**Research Question:** *Can geometric information extracted from reconstructed 3D facial surfaces provide discriminative information for face presentation-attack detection?*

This notebook orchestrates the research workflow by calling repository modules (`src/face_recon_occlusion/`) and scripts (`scripts/`).

---


## SECTION 1 — Environment Setup
Inspect standard libraries, mount Google Drive, and pull the latest code repository.


In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

print(f"Python Version   : {sys.version}")
print(f"Current Directory: {Path.cwd()}")
print(f"Timestamp        : {datetime.now().isoformat()}")


In [ ]:
# Mount Google Drive (Colab Execution)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully at /content/drive")
except ImportError:
    print("ℹ️ Running outside Google Colab environment. Assuming local filesystem storage.")


In [ ]:
# Repository Management
repo_dir = Path("/content/3DFaceReconstruction-With-Occlusion")

if not repo_dir.exists():
    print(f"Cloning repository to {repo_dir}...")
    !git clone https://github.com/manbhavsingh/3DFaceReconstruction-With-Occlusion {repo_dir}
else:
    print(f"Repository already exists at {repo_dir}. Pulling latest main branch...")
    %cd {repo_dir}
    !git pull origin main

%cd {repo_dir}
print(f"✅ Current Working Directory: {Path.cwd()}")


In [ ]:
# Verify Repository Directory Structure
assert (repo_dir / "src").exists(), "Missing src/ directory in repository!"
assert (repo_dir / "scripts").exists(), "Missing scripts/ directory in repository!"
print("✅ Repository layout verified (src/ and scripts/ present).")


## SECTION 2 — Dependency Verification
Verify core scientific dependencies. Do NOT rebuild nvdiffrast or install chumpy.


In [ ]:
import numpy as np
import pandas as pd
import scipy
import trimesh
import sklearn
import matplotlib
import tqdm

print("Dependencies Verified:")
print(f"  numpy        : {np.__version__}")
print(f"  pandas       : {pd.__version__}")
print(f"  scipy        : {scipy.__version__}")
print(f"  trimesh      : {trimesh.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  matplotlib   : {matplotlib.__version__}")
print(f"  tqdm         : {tqdm.__version__}")
print("✅ All required dependencies are available.")


## SECTION 3 — Repository / Package Setup
Ensure `src/` is in `sys.path` and test importing package modules.


In [ ]:
# Add src/ directory to sys.path
src_dir = str(repo_dir / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from face_recon_occlusion.geometry import (
    extract_depth_features,
    extract_gradient_features,
    extract_curvature_features,
    extract_geometry_features
)
from face_recon_occlusion.data import create_subject_disjoint_split, verify_subject_disjoint, summarize_split
from face_recon_occlusion.models import train_svm, build_svm_pipeline
from face_recon_occlusion.evaluation import compute_pad_metrics, run_ablation_experiments, analyze_failures

print("✅ Successfully imported repository modules:")
print("   - face_recon_occlusion.geometry (depth, gradient, curvature, features)")
print("   - face_recon_occlusion.data (splits, metadata summaries)")
print("   - face_recon_occlusion.models (SVM pipeline)")
print("   - face_recon_occlusion.evaluation (PAD metrics, ablation, failure analysis)")


## SECTION 4 — Drive / Data Validation
Validate existing Google Drive dataset path (`/content/drive/MyDrive/BTP_Part2_OULU/subject_disjoint_pilot_100/`), metadata CSV, and 3D OBJ reconstructions.


In [ ]:
drive_root = Path("/content/drive/MyDrive/BTP_Part2_OULU/subject_disjoint_pilot_100")
meta_path = drive_root / "oulu_subject_disjoint_pilot_100.csv"
mesh_dir = drive_root / "reconstructions"

print(f"Checking Drive Root Path : {drive_root} (Exists: {drive_root.exists()})")
print(f"Checking Metadata CSV    : {meta_path} (Exists: {meta_path.exists()})")
print(f"Checking Reconstructions : {mesh_dir} (Exists: {mesh_dir.exists()})")

assert drive_root.exists(), f"Drive root directory not found: {drive_root}"
assert meta_path.exists(), f"Metadata CSV not found: {meta_path}"
assert mesh_dir.exists(), f"Reconstruction directory not found: {mesh_dir}"

obj_files = sorted(list(mesh_dir.rglob("*.obj")))
print(f"\nTotal OBJ Meshes Found  : {len(obj_files)}")
print("Sample Real OBJ Filenames:")
for obj_f in obj_files[:10]:
    print(f"  - {obj_f.name}")

print("\n✅ Drive data validation complete.")


## SECTION 5 — Metadata Inspection
Inspect dataset metadata columns, subject distribution, and class balance.


In [ ]:
meta_df = pd.read_csv(meta_path)
print(f"Metadata Shape: {meta_df.shape}")
print(f"Metadata Columns: {list(meta_df.columns)}\n")
print("First 5 Rows:")
display(meta_df.head())

print("\nSubject Distribution:")
print(f"  Total Unique Subjects: {meta_df['subject_id'].nunique()}")
print(f"  Subject IDs: {sorted(meta_df['subject_id'].unique())}")

print("\nClass Distribution (0 = Bona Fide, 1 = Spoof):")
print(meta_df["label"].value_counts().sort_index())

if "attack_id" in meta_df.columns:
    print("\nAttack Types Distribution:")
    print(meta_df["attack_id"].value_counts())

for col in ["sample_id", "subject_id", "label"]:
    assert col in meta_df.columns, f"CRITICAL: Missing required column '{col}' in metadata CSV!"

print("\n✅ Metadata fields verified successfully.")


## SECTION 6 — SINGLE OBJ GEOMETRY TEST
**Sanity Check:** Test `extract_geometry_features()` on the first actual reconstructed OBJ mesh (e.g. `1_1_36_1_1.obj`). Verify numeric, finite, non-NaN values.


In [ ]:
test_obj_path = obj_files[0]
print(f"Testing Geometry Extraction on Sample OBJ: {test_obj_path.name}")
print(f"Full Path: {test_obj_path}")

test_feats = extract_geometry_features(test_obj_path)

print("\n==========================================")
print("SINGLE OBJ GEOMETRY FEATURE OUTPUT")
print("==========================================")
features_to_check = [
    "depth_std", "depth_range",
    "gradient_mean", "gradient_std",
    "gaussian_curvature_mean", "gaussian_curvature_std"
]

for fname in features_to_check:
    val = test_feats[fname]
    print(f"{fname:25s}: {val:.8f}")
    assert np.isfinite(val), f"CRITICAL FAIL: Feature {fname} is not finite ({val})!"
    assert not np.isnan(val), f"CRITICAL FAIL: Feature {fname} is NaN!"

print("\nNormalization Scale Factors:")
print(f"  xy_scale         : {test_feats['xy_scale']:.6f}")
print(f"  curvature_radius : {test_feats['curvature_radius']:.6f}")
print("\n✅ Single OBJ geometry extraction sanity test PASSED.")


## SECTION 7 — CACHE-FIRST FEATURE EXTRACTION
Run `scripts/run_geometry_extraction.py` batch feature extraction across all OBJ meshes. Reuses valid existing entries and saves progress incrementally.


In [ ]:
output_feature_csv = drive_root / "oulu_geometry_features.csv"

!python scripts/run_geometry_extraction.py \
  --mesh-dir "{mesh_dir}" \
  --metadata-csv "{meta_path}" \
  --output-csv "{output_feature_csv}"

print(f"\n✅ Batch feature extraction completed. Saved to: {output_feature_csv}")


## SECTION 8 — FEATURE CSV VALIDATION
Load and validate the output feature table for expected sample count, feature columns, and absence of NaN/Inf values.


In [ ]:
feats_df = pd.read_csv(output_feature_csv)
print(f"Extracted Feature Table Shape: {feats_df.shape}")
print(f"Columns: {list(feats_df.columns)}\n")

for fname in features_to_check:
    assert fname in feats_df.columns, f"Missing feature column: {fname}"
    nan_cnt = feats_df[fname].isna().sum()
    assert nan_cnt == 0, f"Found {nan_cnt} NaN values in feature {fname}!"

print("Summary Statistics of Extracted Baseline 6 Features:")
display(feats_df[features_to_check].describe())

print("\n✅ Feature CSV validation PASSED.")


## SECTION 9 — SUBJECT-DISJOINT SPLIT VALIDATION
Verify zero subject identity leakage between training and testing splits ($train\_subjects \cap test\_subjects = \emptyset$).


In [ ]:
if "split" not in feats_df.columns:
    feats_df = create_subject_disjoint_split(feats_df, subject_col="subject_id", test_ratio=0.3, random_state=42)

train_subset = feats_df[feats_df["split"] == "train"]
test_subset = feats_df[feats_df["split"] == "test"]

# Explicit verification
verify_subject_disjoint(train_subset, test_subset, subject_col="subject_id")

split_summary = summarize_split(feats_df, subject_col="subject_id", label_col="label")
print("Subject-Disjoint Split Statistics:")
print(f"  Train Samples  : {split_summary['n_train_samples']} (Subjects: {split_summary['n_train_subjects']})")
print(f"  Test Samples   : {split_summary['n_test_samples']} (Subjects: {split_summary['n_test_subjects']})")
print(f"  Train Classes  : {split_summary['train_class_dist']}")
print(f"  Test Classes   : {split_summary['test_class_dist']}")
print("\n✅ Subject-disjoint split verified: ZERO subject identity leakage.")


## SECTION 10 — SVM BASELINE
Train `StandardScaler` + `SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=42)` fitted strictly on training subjects.


In [ ]:
X_train_full = train_subset[features_to_check].values
y_train_full = train_subset["label"].values

X_test_full = test_subset[features_to_check].values
y_test_full = test_subset["label"].values

# Fit scaler and SVM strictly on training split
svm_model = train_svm(X_train_full, y_train_full, C=1.0, kernel="rbf", class_weight="balanced", random_state=42)
print("✅ Baseline RBF SVM pipeline fitted strictly on X_train.")

y_pred_baseline = svm_model.predict(X_test_full)
y_prob_baseline = svm_model.predict_proba(X_test_full)[:, 1]


## SECTION 11 — PAD METRICS
Compute ISO/IEC 30107-3 metrics (APCER, BPCER, ACER, ROC-AUC, Accuracy, Precision, Recall, F1, Confusion Matrix). Label convention: `0 = Bona Fide`, `1 = Spoof`.


In [ ]:
base_metrics = compute_pad_metrics(y_test_full, y_pred_baseline, y_prob_baseline)

print("==========================================")
print("BASELINE 6-FEATURE SVM EVALUATION RESULTS")
print("==========================================")
print(f"Accuracy   : {base_metrics['accuracy']:.4f}")
print(f"APCER      : {base_metrics['apcer']:.4f} (Spoof False Acceptance Rate)")
print(f"BPCER      : {base_metrics['bpcer']:.4f} (Bona Fide False Rejection Rate)")
print(f"ACER       : {base_metrics['acer']:.4f} (Average Classification Error Rate)")
print(f"ROC-AUC    : {base_metrics['roc_auc']:.4f}")
print(f"Precision  : {base_metrics['precision']:.4f}")
print(f"Recall     : {base_metrics['recall']:.4f}")
print(f"F1-Score   : {base_metrics['f1']:.4f}")
print(f"\nConfusion Matrix (TN, FP, FN, TP):")
print(f"  TN={base_metrics['tn']}, FP={base_metrics['fp']}, FN={base_metrics['fn']}, TP={base_metrics['tp']}")


## SECTION 12 — FIVE ABLATION EXPERIMENTS
Execute all 5 required ablation feature combinations using `scripts/run_ablation_experiments.py`.


In [ ]:
ablation_out_dir = drive_root / "ablation_results"

!python scripts/run_ablation_experiments.py \
  --features-csv "{output_feature_csv}" \
  --output-dir "{ablation_out_dir}" \
  --seed 42

ablation_results_df = pd.read_csv(ablation_out_dir / "results.csv")
print("\n==========================================================================================")
print("ABLATION EXPERIMENTS SUMMARY TABLE")
print("==========================================================================================")
display(ablation_results_df[["experiment", "num_features", "accuracy", "apcer", "bpcer", "acer", "roc_auc", "f1"]])
print(f"\n✅ Ablation results saved to: {ablation_out_dir / 'results.csv'}")


## SECTION 13 — FAILURE ANALYSIS
Analyze false positives, false negatives, per-attack error rates, and per-device error rates.


In [ ]:
failures = analyze_failures(
    test_subset, y_pred_baseline, y_prob_baseline,
    subject_col="subject_id",
    attack_col="attack_id" if "attack_id" in test_subset.columns else None,
    device_col="device_id" if "device_id" in test_subset.columns else None
)

print("==========================================")
print("FAILURE ANALYSIS BREAKDOWN")
print("==========================================")
print(f"Total Test Samples      : {failures['n_test_total']}")
print(f"False Positives (Bona Fide misclassified as Spoof) : {failures['n_false_positives']}")
print(f"False Negatives (Spoof misclassified as Bona Fide) : {failures['n_false_negatives']}")
print(f"FP Subject IDs          : {failures['false_positive_subjects']}")
print(f"FN Subject IDs          : {failures['false_negative_subjects']}")

if "per_attack_breakdown" in failures:
    import json
    print("\nPer-Attack Error Breakdown:")
    print(json.dumps(failures["per_attack_breakdown"], indent=2))


## SECTION 14 — GEOMETRY VISUALIZATION
Generate 5-panel diagnostic visualizer for an actual OULU reconstruction sample.


In [ ]:
from face_recon_occlusion.geometry.visualization import visualize_geometry_sample
import matplotlib.pyplot as plt

# Find matching original image and render PNG if available
vis_obj_path = test_obj_path
img_dir = drive_root / "images"
vis_img_path = img_dir / f"{vis_obj_path.stem}.jpg" if (img_dir / f"{vis_obj_path.stem}.jpg").exists() else None
vis_render_path = mesh_dir / f"{vis_obj_path.stem}_render.png" if (mesh_dir / f"{vis_obj_path.stem}_render.png").exists() else None
output_vis_png = drive_root / f"{vis_obj_path.stem}_5panel_vis.png"

fig = visualize_geometry_sample(
    obj_path=vis_obj_path,
    original_img_path=vis_img_path,
    render_img_path=vis_render_path,
    output_path=output_vis_png
)
plt.show()
print(f"✅ Generated 5-panel geometry visualization figure: {output_vis_png}")


## SECTION 15 — FINAL EXPERIMENT SUMMARY
Summary report of dataset statistics, subject splits, baseline SVM metrics, and output file locations.


In [ ]:
print("==========================================================================================")
print("BTP PART 2 RESEARCH EXPERIMENT FINAL SUMMARY")
print("==========================================================================================")
print(f"Dataset Path           : {drive_root}")
print(f"Total Valid Samples    : {len(feats_df)}")
print(f"Total Unique Subjects  : {feats_df['subject_id'].nunique()}")
print(f"Train / Test Split     : {split_summary['n_train_samples']} train ({split_summary['n_train_subjects']} subs) | {split_summary['n_test_samples']} test ({split_summary['n_test_subjects']} subs)")
print(f"Subject Leakage        : NONE (verify_subject_disjoint passed)")
print(f"Baseline 6 Features    : depth_std, depth_range, gradient_mean, gradient_std, gaussian_curvature_mean, gaussian_curvature_std")
print(f"Baseline SVM ACER      : {base_metrics['acer']:.4f} (APCER: {base_metrics['apcer']:.4f}, BPCER: {base_metrics['bpcer']:.4f})")
print(f"Baseline SVM ROC-AUC   : {base_metrics['roc_auc']:.4f}")
print(f"Generated Files        :")
print(f"  1. Feature CSV        : {output_feature_csv}")
print(f"  2. Ablation Config    : {ablation_out_dir / 'experiment_config.json'}")
print(f"  3. Ablation Summary   : {ablation_out_dir / 'results.csv'}")
print(f"  4. Sample Visualizer  : {output_vis_png}")
print("==========================================================================================")
